In [ ]:
import pandas as pd
import altair as alt
from scipy import stats

# Hardcoded file paths

In [ ]:
sge_file = './Data/final_tables/supplementary_file_1_BARD1_SGE_final_table.xlsx'

thermompnn_files = {
    'RING':'./Data/extra_data/ThermoMPNN_data/BARD1_RING.csv',
    'ARD':'./Data/extra_data/ThermoMPNN_data/BARD1_ARD.csv', #need to drop first 3 residues in the CSV
    'BRCT':'./Data/extra_data/ThermoMPNN_data/BARD1_BRCT.csv'
}

thermompnn_offsets ={
    'RING':26,
    'ARD': -2,
    'BRCT': 568
}

In [ ]:
def aa_spliter(df,aa_mut_col='amino_acid_change', ref_aa_col='ref_aa', aa_pos_col='aa_pos', alt_aa_col='alt_aa'):
    df[ref_aa_col] = df[aa_mut_col].transform(lambda x: x[0])
    df[alt_aa_col] = df[aa_mut_col].transform(lambda x: x[-1])
    df[aa_pos_col]=df[aa_mut_col].transform(lambda x: int(x[1:-1]))

    return df

# Dataframe processing and merging

## SGE df processing

In [ ]:
raw_sge_df = pd.read_excel(sge_file, sheet_name='scores')

sge_df = raw_sge_df[(raw_sge_df['amino_acid_change']!='---') & (raw_sge_df['var_type']=='snv')].copy()
sge_df =sge_df[['pos', 'ref', 'alt', 'exon', 'target', 'consequence', 'score', 'functional_consequence', 'amino_acid_change', 'pos_id', 'RNAscore', 'RNA_consequence']]
sge_df = aa_spliter(sge_df)

sge_df = sge_df[~sge_df['consequence'].isin(['stop_lost', 'stop_gained'])]

## ThermoMPNN processing

In [ ]:
domains = list(thermompnn_files.keys())

thermompnn_dfs = []

for domain in domains:
    thermompnn_df = pd.read_csv(thermompnn_files[domain])

    if domain =='ARD':
        thermompnn_df = thermompnn_df[thermompnn_df['pos']>100].copy()

    thermompnn_df['pos'] = thermompnn_df['pos'] + thermompnn_offsets[domain]
    thermompnn_df['amino_acid_change'] = thermompnn_df['wtAA'] + thermompnn_df['pos'].astype(str) + thermompnn_df['mutAA']

    thermompnn_df = thermompnn_df.rename(columns={'pos': 'aa_pos',
                                                    'wtAA': 'ref_aa',
                                                    'mutAA': 'alt_aa',
                                                    'ddG (kcal/mol)': 'ddG'})
    
    thermompnn_df['domain'] = domain
    thermompnn_df = thermompnn_df[['domain', 'amino_acid_change', 'ref_aa', 'alt_aa', 'aa_pos', 'ddG']]
    thermompnn_dfs.append(thermompnn_df)



final_thermompnn_df = pd.concat(thermompnn_dfs)
final_thermompnn_df

## Merge dataframes

In [ ]:
final_df = pd.merge(sge_df, final_thermompnn_df, on=['amino_acid_change', 'ref_aa', 'alt_aa', 'aa_pos'], how='inner')

final_df

# Normal vs. LoF ddG comparison

In [ ]:
missense_only = final_df[(final_df['consequence']=='missense_variant') & (final_df['functional_consequence'].isin(['functionally_normal', 'functionally_abnormal'])) & (final_df['RNA_consequence']=='normal')]

boxplot = alt.Chart(missense_only).mark_boxplot(size=50).encode(
    x=alt.X('functional_consequence:N'),
    y=alt.Y('ddG:Q'),
    tooltip=['amino_acid_change','ddG', 'score']
).properties(
    height=300,
    width=200
).interactive()

boxplot.display()

_, p_val = stats.mannwhitneyu(missense_only[missense_only['functional_consequence']=='functionally_abnormal']['ddG'].tolist(), 
                         missense_only[missense_only['functional_consequence']=='functionally_normal']['ddG'].tolist())

print(f'The Mann-Whitney p-value is: {p_val}')

# Heatmap visualization

In [ ]:
# Universal font sizes — change these to resize all axes and legend labels at once
x_label_size = 22      # Font size for x-axis tick labels
x_title_size = 24      # Font size for x-axis titles
y_label_size = 22      # Font size for y-axis tick labels
y_title_size = 24      # Font size for y-axis titles
legend_label_size = 20 # Font size for legend item labels
legend_title_size = 22 # Font size for legend titles

def heatmap(df, score_col='score', score_name='Fitness Score', map_domain=[-0.2,0], reverse_colors=True):
    order = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y', 'Stop', 'Min.', 'Mean']

    height_per_category = 20

    #df = df.loc[~(df['AAsub'].isin(['*778*']))] #Gets rid of stop-loss variant
    
    map = alt.Chart(df).mark_rect().encode(
        x = alt.X('aa_pos:Q',
                  title = '',
                  axis = alt.Axis(values=list(range(0, 778, 25))),
                  scale = alt.Scale(domain = [0,778], zero=False),
                  bin = alt.Bin(maxbins = 778, minstep = 1)
                 ),
        y = alt.Y('alt_aa',
                  title = 'Amino Acid Substitution',
                  axis = alt.Axis(
                      labelFontSize = y_label_size,
                      titleFontSize = y_title_size
                  ),
                 sort = order),
        color = alt.Color(score_col, 
                          title = score_name,
                          scale = alt.Scale(
                              domain = map_domain,
                              clamp = True,
                              scheme = 'bluepurple',
                              reverse = reverse_colors
                          ),
                          legend = alt.Legend(
                              titleFontSize = legend_title_size,
                              labelFontSize = legend_label_size
                          )
                         ),
        tooltip=['amino_acid_change','score', 'ddG']
    ).properties(
        height = height_per_category * len(order), 
        width = 1450
    )

    return map

In [ ]:
sge_map = heatmap(final_df)

In [ ]:
ddg_map = heatmap(final_df, score_col='ddG', score_name='ddG', map_domain=[0, 3], reverse_colors=False)

In [ ]:
final_map = (sge_map & ddg_map).resolve_scale(color='independent').interactive()

final_map

In [ ]:
lof_only = final_df[final_df['functional_consequence']=='functionally_abnormal']
lof_ddG_map = heatmap(lof_only, score_col='ddG', score_name='ddG', map_domain=[0, 2], reverse_colors=False)

full_lof_map = (sge_map & lof_ddG_map).interactive().resolve_scale(color='independent')

full_lof_map.display()